In [0]:
Delta Lake does not use traditional B-Tree indexes. Instead, it relies on data skipping using file-level statistics, partition pruning, and techniques like Z-ordering and bloom filters to reduce the amount of data scanned and improve query performance.

In [0]:
>>Delta Lake does NOT use traditional B-Tree indexes (clustered/non-clustered) for query acceleration.
>>Instead, it uses a combination of data skipping + layout optimizations.
>>What replaces indexes in Delta Lake

1. Data Skipping (Min/Max Statistics)
Each file stores metadata: (min value),(max value),(null count)
Stored in the Delta transaction log.
👉 Query:
SELECT * FROM table WHERE id = 100;
Engine:
Skips files where 100 is not in [min, max]
This is the closest thing to an “index”

2. Z-Ordering (Like multi-column indexing)
OPTIMIZE table ZORDER BY (user_id, date);
Co-locates related data in same files
Reduces number of files scanned
👉 Similar to:
composite index (but at file level, not row level)

3. Partitioning
PARTITIONED BY (date)
Physically separates data into folders
Query prunes partitions
👉 Similar to:
coarse-grained index

4. Bloom Filter Index (optional feature)
CREATE BLOOMFILTER INDEX ON TABLE table
FOR COLUMNS(id OPTIONS (fpp = 0.1, numItems = 1000000));
Helps with point lookups
Avoids scanning irrelevant files
👉 Closest to:
probabilistic index (not exact like B-Tree)

5. File-level parallelism
Many small files → parallel reads
Combined with skipping → fast queries
How it works in Unity Catalog (Databricks)
When you query:

SELECT * FROM catalog.schema.table WHERE user_id = 123;
Execution flow:
Check transaction log metadata
Skip irrelevant files (data skipping)
Apply partition pruning
Use Z-order layout if present
Scan only relevant files
Key difference vs traditional DB

Feature	Traditional DB	Delta Lake

Index type	B-Tree	Metadata + layout

Granularity	Row-level	File-level
Maintenance	Automatic	Needs OPTIMIZE / design
Best for	OLTP	OLAP / big data

When to use what
Partitioning → large filters (date, region)
Z-ORDER → frequent query columns
Bloom filter → point lookup
OPTIMIZE → reduce small files

In [0]:
🔹 1. File size optimization (OPTIMIZE)
Avoid too many small files (major bottleneck)

Use:

OPTIMIZE table_name;

For large tables:

OPTIMIZE table_name ZORDER BY (column);

👉 Target ~128MB–1GB file size

🔹 2. Z-Ordering (data skipping)
Co-locates related data in same files
Helps queries with filters
OPTIMIZE table_name ZORDER BY (user_id, date);

👉 Best for high-cardinality columns used in filters

🔹 3. Partitioning (use carefully)
Partition by low-cardinality columns (e.g., date)
PARTITIONED BY (date)

⚠️ Avoid over-partitioning → leads to small files

🔹 4. Data skipping & statistics
Delta automatically stores min/max stats
Improves query pruning without scanning full data
👉 Works best when columns are well distributed
🔹 5. Auto Optimize & Auto Compaction
Enable at table/session level:
SET spark.databricks.delta.autoOptimize.optimizeWrite = true;
SET spark.databricks.delta.autoOptimize.autoCompact = true;

👉 Prevents small file problem during writes

🔹 6. Vacuum (storage cleanup)
Removes old unused files
VACUUM table_name RETAIN 168 HOURS;

👉 Reduces storage cost

🔹 7. Caching (for repeated queries)
CACHE TABLE table_name;

👉 Useful for BI/dashboard workloads

🔹 8. Predicate pushdown
Always filter early:
WHERE date = '2025-01-01'

👉 Reduces data scanned

🔹 9. Schema optimization
Avoid wide tables
Use proper data types (no unnecessary strings)
👉 Reduces memory + improves scan speed
🔹 10. Merge optimization (for upserts)
Use partition pruning + conditions in MERGE
Avoid full table scans in upserts
🔹 Interview-ready short answer

Use OPTIMIZE (compaction), Z-ORDER for data skipping, proper partitioning, auto-compaction, caching, and predicate pushdown to reduce file scans and improve Delta Lake performance in production.